<a href="https://colab.research.google.com/github/mbilalerkoc/Medical-Image-Segmentation-System/blob/main/Kidney_U_Net_E%C4%9Fitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from google.colab import drive

print("⏳ Google Drive bağlanıyor...")
drive.mount('/content/drive')

# ==============================================================================
# 1. ADIM: KONFİGÜRASYON VE MERKEZİ DOSYA YOLLARI (U-NET İÇİN)
# ==============================================================================
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 75
LEARNING_RATE = 1e-3
TEST_ORANI = 0.15
VAL_ORANI = 0.15
RANDOM_SEED = 42

# --- ANA DİZİNLER ---
DRIVE_BASE = '/content/drive/MyDrive/Medical-Image-Segmentation-System'
CIKARTILACAK_YER = '/content'

KIDNEY_ENGINE_DIR = f'{DRIVE_BASE}/ai_engine_kidney'
MODEL_KAYIT        = f'{KIDNEY_ENGINE_DIR}/saved_models'
SONUC_KLASOR       = f'{KIDNEY_ENGINE_DIR}/results'
PROCESSED_DATA_DIR = f'{KIDNEY_ENGINE_DIR}/processed_data'

for p in [MODEL_KAYIT, SONUC_KLASOR, PROCESSED_DATA_DIR]:
    os.makedirs(p, exist_ok=True)

# 🎯 --- MERKEZİ DOSYA İSİMLENDİRME (STANDART U-NET) ---
MODEL_EN_IYI_YOL = f"{MODEL_KAYIT}/bobrek_U-Net_en_iyi.h5"
MODEL_FINAL_YOL  = f"{MODEL_KAYIT}/bobrek_U-Net_final.h5"

NPY_X_YOL = f"{PROCESSED_DATA_DIR}/bobrek_U-Net_X.npy"
NPY_Y_YOL = f"{PROCESSED_DATA_DIR}/bobrek_U-Net_Y.npy"

GRAFIK_YOL = f"{SONUC_KLASOR}/bobrek_U-Net_egitim_grafikleri.png"
TAHMIN_YOL = f"{SONUC_KLASOR}/bobrek_U-Net_ornek_tahminler.png"
METRIK_YOL = f"{SONUC_KLASOR}/bobrek_U-Net_test_metrikleri.txt"

print(f"✅ Klasör yapısı 'ai_engine_kidney' altında organize edildi (U-NET)\n")

# 🎯 AKILLI KLASÖR BULUCU (archive.zip için)
IMG_DIR, MASK_DIR = None, None
for root, dirs, files in os.walk(CIKARTILACAK_YER):
    if 'drive' in root: continue
    for d in dirs:
        if d.lower() in ['images', 'image']: IMG_DIR = os.path.join(root, d)
        if d.lower() in ['masks', 'mask']: MASK_DIR = os.path.join(root, d)

if not (IMG_DIR and MASK_DIR):
    zip_yolu = f'{DRIVE_BASE}/dataset/archive.zip'
    print("⏳ archive.zip doğrudan ana dizine (/content) çıkarılıyor...")
    !unzip -q -o "{zip_yolu}" -d "{CIKARTILACAK_YER}"
    for root, dirs, files in os.walk(CIKARTILACAK_YER):
        if 'drive' in root: continue
        for d in dirs:
            if d.lower() in ['images', 'image']: IMG_DIR = os.path.join(root, d)
            if d.lower() in ['masks', 'mask','label']: MASK_DIR = os.path.join(root, d)

if IMG_DIR and MASK_DIR:
    print(f"🔍 Bulunan Resim Klasörü: {IMG_DIR}")
    print(f"🔍 Bulunan Maske Klasörü: {MASK_DIR}\n")
else:
    raise FileNotFoundError("❌ HATA: Zip dosyası çıkarıldı ama resim/maske klasörleri bulunamadı!")

# ==============================================================================
# 2. ADIM: VERİ ÖN İŞLEME VE ARTIRMA
# ==============================================================================
def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def preprocess(img_path, mask_path):
    img = cv2.imread(img_path)
    # OpenCV, .tif veya .png fark etmeksizin maskeyi gri tonda okur
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None: return None, None

    img = apply_clahe(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0

    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    mask = mask.astype(np.float32) / 255.0

    if len(mask.shape) == 2:
        mask = np.expand_dims(mask, axis=-1)
    return img, mask

def augment(img, mask):
    if np.random.rand() > 0.5:
        img = cv2.flip(img, 1)
        mask = cv2.flip(mask, 1)
    if np.random.rand() > 0.7:
        img = cv2.flip(img, 0)
        mask = cv2.flip(mask, 0)
    if np.random.rand() > 0.7:
        angle = np.random.uniform(-10, 10)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h))
        mask = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST)

    if len(mask.shape) == 2:
        mask = np.expand_dims(mask, axis=-1)
    return img, mask

# ==============================================================================
# 3. ADIM: DATA GENERATOR SİSTEMİ (AKILLI EŞLEŞTİRME)
# ==============================================================================
def veri_yukle_ve_ayir():
    img_paths, mask_paths = [], []
    # 🎯 .tif ve .jpeg uzantılarını listeye ekledik
    images_list = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.png', '.tif', '.jpeg'))])

    for f in images_list:
        stem = os.path.splitext(f)[0]
        # 🎯 Kaggle maske isimlendirme formatlarının hepsini tek tek dener
        olasi_maskeler = [f"{stem}.png", f"{stem}.tif", f"{stem}_mask.png", f"{stem}_mask.tif"]

        for m_ad in olasi_maskeler:
            if os.path.exists(os.path.join(MASK_DIR, m_ad)):
                img_paths.append(os.path.join(IMG_DIR, f))
                mask_paths.append(os.path.join(MASK_DIR, m_ad))
                break # Eşleşmeyi bulduğu an diğer olası isimleri aramayı bırakır

    print(f"✅ Eşleşen Toplam Veri Sayısı: {len(img_paths)}")
    if len(img_paths) == 0:
        raise ValueError("❌ HATA: Eşleşen resim ve maske bulunamadı. Veri seti bomboş!")

    X_temp, X_test, y_temp, y_test = train_test_split(img_paths, mask_paths, test_size=TEST_ORANI, random_state=RANDOM_SEED)
    val_oran_hesabi = VAL_ORANI / (1.0 - TEST_ORANI)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_oran_hesabi, random_state=RANDOM_SEED)

    print(f"📊 Dağılım -> Eğitim: {len(X_train)} | Validasyon: {len(X_val)} | Test: {len(X_test)}")
    return X_train, X_val, X_test, y_train, y_val, y_test

def test_verilerini_npy_kaydet(X_test_paths, y_test_paths):
    print("\n📦 Test verileri VS Code (.npy) için işleniyor...")
    X_test_islenmis, y_test_islenmis = [], []
    for x_p, y_p in zip(X_test_paths, y_test_paths):
        img, mask = preprocess(x_p, y_p)
        if img is not None:
            X_test_islenmis.append(img)
            y_test_islenmis.append(mask)

    X_np = np.array(X_test_islenmis, dtype=np.float32)
    Y_np = np.array(y_test_islenmis, dtype=np.float32)

    np.save(NPY_X_YOL, X_np)
    np.save(NPY_Y_YOL, Y_np)
    print(f"✅ Test verileri kaydedildi: {PROCESSED_DATA_DIR}")
    return X_np, Y_np

class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, img_paths, mask_paths, batch_size, is_train=True):
        self.img_paths = np.array(img_paths)
        self.mask_paths = np.array(mask_paths)
        self.batch_size = batch_size
        self.is_train = is_train

    def __len__(self):
        return int(np.ceil(len(self.img_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_x = self.img_paths[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_y = self.mask_paths[idx * self.batch_size : (idx + 1) * self.batch_size]
        X, Y = [], []
        for x_p, y_p in zip(batch_x, batch_y):
            img, mask = preprocess(x_p, y_p)
            if img is not None:
                if self.is_train and np.random.rand() > 0.5:
                    img, mask = augment(img.copy(), mask.copy())
                X.append(img)
                Y.append(mask)
        return np.array(X), np.array(Y)

    def on_epoch_end(self):
        if self.is_train:
            indices = np.arange(len(self.img_paths))
            np.random.shuffle(indices)
            self.img_paths = self.img_paths[indices]
            self.mask_paths = self.mask_paths[indices]

# ==============================================================================
# 4. ADIM: LOSS FONKSİYONLARI VE STANDART U-NET MİMARİSİ
# ==============================================================================
def dice_katsayisi(y_gercek, y_tahmin):
    y_gercek_duz = tf.reshape(y_gercek, [-1])
    y_tahmin_duz = tf.reshape(y_tahmin, [-1])
    kesisim = tf.reduce_sum(y_gercek_duz * y_tahmin_duz)
    return (2. * kesisim + 1e-5) / (tf.reduce_sum(y_gercek_duz) + tf.reduce_sum(y_tahmin_duz) + 1e-5)

def dice_loss(y_gercek, y_tahmin):
    return 1.0 - dice_katsayisi(y_gercek, y_tahmin)

def agirlikli_birlesik_loss(y_gercek, y_tahmin):
    bce = tf.keras.losses.binary_crossentropy(y_gercek, y_tahmin)
    return (0.7 * bce) + (0.3 * dice_loss(y_gercek, y_tahmin))

def conv_block(x, filters):
    x = layers.Conv2D(filters, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return x

# 🎯 DÜZELTİLDİ: Sadece Standart U-Net
def unet_modeli_kur(giris_boyutu=(IMG_SIZE, IMG_SIZE, 3)):
    girdiler = Input(giris_boyutu)

    # ─── ENCODER ───
    c1 = conv_block(girdiler, 32)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = conv_block(p3, 256)
    p4 = layers.MaxPooling2D((2, 2))(c4)

    # ─── BOTTLE-NECK ───
    c5 = conv_block(p4, 512)

    # ─── DECODER ───
    u6 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = conv_block(u6, 256)

    u7 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = conv_block(u7, 128)

    u8 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = conv_block(u8, 64)

    u9 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = conv_block(u9, 32)

    cikti = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)

    model = models.Model(inputs=girdiler, outputs=cikti)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss=agirlikli_birlesik_loss, metrics=['accuracy', dice_katsayisi])
    return model

# ==============================================================================
# 5. ADIM: RAPORLAMA VE EVALUATE
# ==============================================================================
def sonuclari_degerlendir_ve_kaydet(model, test_generator, history):
    print("\n📊 Değerlendirme ve Raporlama Aşaması Başlıyor...")

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Eğitim Loss', color='#22c55e')
    plt.plot(history.history['val_loss'], label='Doğrulama Loss', color='#ef4444')
    plt.title('Böbrek Taşı (U-NET) - Model Kayıp (Loss)')
    plt.legend()

    if 'dice_katsayisi' in history.history:
        plt.subplot(1, 2, 2)
        plt.plot(history.history['dice_katsayisi'], label='Eğitim Dice', color='#3b82f6')
        plt.plot(history.history['val_dice_katsayisi'], label='Doğrulama Dice', color='#f59e0b')
        plt.title('Böbrek Taşı (U-NET) - Segmentasyon Başarısı')
        plt.legend()

    plt.tight_layout()
    plt.savefig(GRAFIK_YOL)
    plt.close()

    TP = FP = FN = 0
    esik = 0.80

    for i in range(len(test_generator)):
        X_batch, y_batch = test_generator[i]
        tahminler = model.predict(X_batch, verbose=0)
        tahmin_binary = (tahminler > esik).astype(np.float32)
        gercek = y_batch.astype(np.float32)
        TP += np.sum(tahmin_binary * gercek)
        FP += np.sum(tahmin_binary * (1 - gercek))
        FN += np.sum((1 - tahmin_binary) * gercek)

    precision = TP / (TP + FP + 1e-7)
    recall    = TP / (TP + FN + 1e-7)
    iou       = TP / (TP + FP + FN + 1e-7)

    test_sonuclari = model.evaluate(test_generator, verbose=0)
    test_loss = test_sonuclari[0]
    test_dice = test_sonuclari[2] if len(test_sonuclari) > 2 else 0.0

    sonuc_metni = f"""
╔══════════════════════════════════════════╗
║     BÖBREK TAŞI (STANDART U-NET)         ║
╠══════════════════════════════════════════╣
║  Dice Score: {test_dice:.4f}                     ║
║  IoU       : {iou:.4f}                     ║
║  Precision : {precision:.4f}                     ║
║  Recall    : {recall:.4f}                     ║
║  Loss      : {test_loss:.4f}                     ║
╚══════════════════════════════════════════╝
    """
    print(sonuc_metni)

    with open(METRIK_YOL, 'w', encoding='utf-8') as f:
        f.write(sonuc_metni)

    print("🎨 Örnek maske tahminleri görselleştiriliyor...")
    ornek_X, ornek_Y, ornek_Tahmin = [], [], []
    for i in range(len(test_generator)):
        X_batch, Y_batch = test_generator[i]
        tahminler_batch = model.predict(X_batch, verbose=0)
        for j in range(len(Y_batch)):
            if Y_batch[j].max() > 0:
                ornek_X.append(X_batch[j])
                ornek_Y.append(Y_batch[j])
                ornek_Tahmin.append(tahminler_batch[j])
                if len(ornek_X) == 5: break
        if len(ornek_X) == 5: break

    if len(ornek_X) > 0:
        plt.figure(figsize=(15, 9))
        for i in range(len(ornek_X)):
            plt.subplot(3, 5, i + 1)
            plt.imshow(ornek_X[i].squeeze(), cmap='gray')
            plt.axis('off')
            plt.subplot(3, 5, i + 6)
            plt.imshow(ornek_Y[i].squeeze(), cmap='gray')
            plt.axis('off')
            plt.subplot(3, 5, i + 11)
            plt.imshow(ornek_Tahmin[i].squeeze() > esik, cmap='gray')
            plt.axis('off')
        plt.tight_layout()
        plt.savefig(TAHMIN_YOL, dpi=150)
        plt.close()
        print(f"✅ Analizler tamamlandı! Çıktılar Drive'da ({SONUC_KLASOR}).")

# ==============================================================================
# 6. ADIM: TETİĞİ ÇEKME (MAIN)
# ==============================================================================
print("\n" + "="*60 + "\n 🚀 BÖBREK TAŞI (STANDART U-NET) EĞİTİMİ BAŞLIYOR\n" + "="*60)

X_train_p, X_val_p, X_test_p, y_train_p, y_val_p, y_test_p = veri_yukle_ve_ayir()
X_test_npy, y_test_npy = test_verilerini_npy_kaydet(X_test_p, y_test_p)

train_generator = DataGenerator(X_train_p, y_train_p, BATCH_SIZE, is_train=True)
val_generator = DataGenerator(X_val_p, y_val_p, BATCH_SIZE, is_train=False)
test_generator = DataGenerator(X_test_p, y_test_p, BATCH_SIZE, is_train=False)

model = unet_modeli_kur()

callbacks = [
    ModelCheckpoint(MODEL_EN_IYI_YOL, monitor='val_loss', save_best_only=True, mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=20, mode='min', restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=1)
]

history = model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS, callbacks=callbacks)

model.save(MODEL_FINAL_YOL)
sonuclari_degerlendir_ve_kaydet(model, test_generator, history)

⏳ Google Drive bağlanıyor...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Klasör yapısı 'ai_engine_kidney' altında organize edildi (U-NET)

⏳ archive.zip doğrudan ana dizine (/content) çıkarılıyor...
🔍 Bulunan Resim Klasörü: /content/data/image
🔍 Bulunan Maske Klasörü: /content/data/label


 🚀 BÖBREK TAŞI (STANDART U-NET) EĞİTİMİ BAŞLIYOR
✅ Eşleşen Toplam Veri Sayısı: 838
📊 Dağılım -> Eğitim: 586 | Validasyon: 126 | Test: 126

📦 Test verileri VS Code (.npy) için işleniyor...
✅ Test verileri kaydedildi: /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/processed_data


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/75
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 564ms/step - accuracy: 0.6433 - dice_katsayisi: 0.0012 - loss: 0.8005
Epoch 1: val_loss improved from None to 0.67721, saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5



Epoch 1: finished saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 59s 801ms/step - accuracy: 0.7893 - dice_katsayisi: 0.0014 - loss: 0.7490 - val_accuracy: 0.9994 - val_dice_katsayisi: 0.0012 - val_loss: 0.6772 - learning_rate: 0.0010
Epoch 2/75
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.9921 - dice_katsayisi: 0.0015 - loss: 0.6565
Epoch 2: val_loss improved from 0.67721 to 0.57493, saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5



Epoch 2: finished saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 140ms/step - accuracy: 0.9959 - dice_katsayisi: 0.0015 - loss: 0.6312 - val_accuracy: 0.9994 - val_dice_katsayisi: 0.0012 - val_loss: 0.5749 - learning_rate: 0.0010
Epoch 3/75
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.9992 - dice_katsayisi: 0.0015 - loss: 0.5553 
Epoch 3: val_loss improved from 0.57493 to 0.48883, saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5


...


Epoch 56: finished saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 178ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9322 - loss: 0.0209 - val_accuracy: 0.9999 - val_dice_katsayisi: 0.9391 - val_loss: 0.0187 - learning_rate: 5.0000e-04
Epoch 57/75
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9378 - loss: 0.0192
Epoch 57: val_loss did not improve from 0.01871
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 124ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9360 - loss: 0.0198 - val_accuracy: 0.9999 - val_dice_katsayisi: 0.9356 - val_loss: 0.0198 - learning_rate: 5.0000e-04
Epoch 58/75
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9383 - loss: 0.0190
Epoch 58: val_loss did not improve from 0.01871
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 123ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9348 - loss: 0.0200 - val_accuracy: 0.9999 - val_dice_ka


Epoch 64: finished saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 180ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9378 - loss: 0.0190 - val_accuracy: 0.9999 - val_dice_katsayisi: 0.9391 - val_loss: 0.0187 - learning_rate: 2.5000e-04
Epoch 65/75
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9374 - loss: 0.0193
Epoch 65: val_loss improved from 0.01869 to 0.01854, saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5



Epoch 65: finished saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 136ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9394 - loss: 0.0187 - val_accuracy: 0.9999 - val_dice_katsayisi: 0.9397 - val_loss: 0.0185 - learning_rate: 2.5000e-04
Epoch 66/75
36/37 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9364 - loss: 0.0196
Epoch 66: val_loss did not improve from 0.01854
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 122ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9371 - loss: 0.0194 - val_accuracy: 0.9999 - val_dice_katsayisi: 0.9339 - val_loss: 0.0204 - learning_rate: 2.5000e-04
Epoch 67/75
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9355 - loss: 0.0199
Epoch 67: val_loss improved from 0.01854 to 0.01820, saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5



Epoch 67: finished saving model to /content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/saved_models/bobrek_U-Net_en_iyi.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 141ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9391 - loss: 0.0187 - val_accuracy: 0.9999 - val_dice_katsayisi: 0.9409 - val_loss: 0.0182 - learning_rate: 2.5000e-04
Epoch 68/75
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9419 - loss: 0.0179
Epoch 68: val_loss did not improve from 0.01820
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 124ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9387 - loss: 0.0189 - val_accuracy: 0.9999 - val_dice_katsayisi: 0.9070 - val_loss: 0.0284 - learning_rate: 2.5000e-04
Epoch 69/75
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9343 - loss: 0.0200
Epoch 69: val_loss did not improve from 0.01820
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 125ms/step - accuracy: 0.9999 - dice_katsayisi: 0.9397 - loss: 0.0184 - val_accuracy: 0.9999 - val_dice_ka


📊 Değerlendirme ve Raporlama Aşaması Başlıyor...

╔══════════════════════════════════════════╗
║     BÖBREK TAŞI (STANDART U-NET)         ║
╠══════════════════════════════════════════╣
║  Dice Score: 0.9440                     ║
║  IoU       : 0.9043                     ║
║  Precision : 0.9460                     ║
║  Recall    : 0.9535                     ║
║  Loss      : 0.0171                     ║
╚══════════════════════════════════════════╝
    
🎨 Örnek maske tahminleri görselleştiriliyor...
✅ Analizler tamamlandı! Çıktılar Drive'da (/content/drive/MyDrive/Medical-Image-Segmentation-System/ai_engine_kidney/results).
